# Model Quantization with SageMaker Processing

This notebook demonstrates how to quantize a Hugging Face model using SageMaker Processing jobs. We'll use the ONNX Runtime and Optimum library to perform quantization.

## What is Quantization?

Quantization is one of the most effective techniques for reducing model footprint with minimal performance degradation. It works by representing model weights and/or activations with lower precision numerical formats.

### Precision Formats

| Format | Bits | Size Reduction | Accuracy Impact | Notes |
|--------|------|----------------|-----------------|-------|
| FP32 | 32-bit | Baseline | None (reference) | Standard training format |
| FP16 | 16-bit | ~2x | Minimal | Good balance for most models |
| BF16 | 16-bit | ~2x | Minimal | Better numerical stability than FP16 |
| INT8 | 8-bit | ~4x | Low-Moderate | Standard for production deployment |
| INT4 | 4-bit | ~8x | Moderate | Emerging standard for large models |

### Types of Quantization

1. **Post-Training Quantization (PTQ)**: Converts model weights to lower precision after training is complete
2. **Quantization-Aware Training (QAT)**: Simulates quantization effects during training to minimize accuracy loss

In this notebook, we'll focus on Post-Training Quantization using dynamic quantization to INT8, which offers a good balance between size reduction and accuracy preservation.

## Setup

First, let's import the necessary libraries and set up our SageMaker session.

In [ ]:
# Import required libraries
import os
import sys
import time
import json
import boto3
import sagemaker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load stored variables from notebook 1
%store -r S3_BUCKET
%store -r SAGEMAKER_ROLE_ARN
%store -r AWS_REGION
%store -r OPTIMIZATION_INSTANCE_TYPE

# Create our own sagemaker session
sagemaker_session = sagemaker.Session()

# Import specific modules for this notebook
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
from sagemaker.huggingface import HuggingFaceModel

# Verify variables were loaded
print(f"Loaded variables:")
print(f"S3_BUCKET: {S3_BUCKET}")
print(f"AWS_REGION: {AWS_REGION}")
print(f"OPTIMIZATION_INSTANCE_TYPE: {OPTIMIZATION_INSTANCE_TYPE}")


## Load Model Information

Let's load the model information from the previous notebooks.

In [ ]:
# Try to load model_info.json if it exists
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded model information from model_info.json")
except FileNotFoundError:
    # If file doesn't exist, create a default model info that matches what setup notebook uploads
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased"
        }
    }

# For this notebook, we'll focus on the sentiment analysis model
model_key = "sentiment-analysis"  # Using hyphen instead of underscore
model_data = model_info[model_key]
model_name = model_data["model_name"]
model_s3_uri = model_data.get("s3_uri", f"s3://{S3_BUCKET}/models/{model_name.replace('/', '-')}")  # Using hyphens

print(f"Using model: {model_name}")
print(f"Model S3 URI: {model_s3_uri}")

## Configure Quantization Processing Job

Now, let's configure and run a SageMaker Processing job to quantize our model.

In [ ]:
# Configure the processing job
processor = PyTorchProcessor(
    framework_version='2.4.0',
    py_version='py311',
    role=SAGEMAKER_ROLE_ARN,
    instance_count=1,
    instance_type=OPTIMIZATION_INSTANCE_TYPE,  # Use the instance type defined in notebook 1
    base_job_name=f'quantize-{model_name.replace("/", "-")}',
    sagemaker_session=sagemaker_session
)

In [ ]:
# Run the processing job with explicit wait control
processor.run(
    code='quantization_script.py',  # Just the filename
    source_dir='scripts',           # Directory containing script and requirements.txt
    inputs=[
        ProcessingInput(
            source=model_s3_uri,
            destination='/opt/ml/processing/input/model'
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name='quantized_model',
            source='/opt/ml/processing/output'
        )
    ],
    arguments=[
        '--quantization-approach', 'dynamic',
        '--bits', '8'
    ],
    wait=True,  # Explicitly set wait behavior
    logs=True   # Show logs during execution
)

## Get Quantized Model S3 Path

After the processing job completes, let's get the S3 path to the quantized model.

In [ ]:
# Get the processing job name
processing_job_name = processor.latest_job.job_name
print(f"Processing job name: {processing_job_name}")

# Get the output S3 URI
processing_job_description = sagemaker_session.sagemaker_client.describe_processing_job(
    ProcessingJobName=processing_job_name
)
output_s3_uri = processing_job_description['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
print(f"Output S3 URI: {output_s3_uri}")

# Define the quantized model name
quantized_model_name = f"{model_name.replace('/', '-')}-quantized"
print(f"Quantized model name: {quantized_model_name}")

## Prepare Quantized Model for Deployment

Now, let's prepare the quantized model for deployment by creating a model.tar.gz file. We'll also include a custom inference script to handle the ONNX model properly.

In [ ]:
# Create a temporary directory to store the model files
import tempfile
import shutil
import tarfile

with tempfile.TemporaryDirectory() as tmpdirname:
    # Download the quantized model from S3
    print(f"Downloading quantized model from {output_s3_uri}...")
    !aws s3 cp --recursive {output_s3_uri} {tmpdirname}/model/
    
    # Create code directory for inference script
    os.makedirs(f"{tmpdirname}/model/code", exist_ok=True)
    
    # Copy the inference script to the model directory
    print(f"Adding custom inference script...")
    shutil.copy('scripts/inference.py', f"{tmpdirname}/model/code/inference.py")
    
    # Copy the requirements file for the inference script
    print(f"Adding inference requirements...")
    shutil.copy('scripts/inference_requirements.txt', f"{tmpdirname}/model/code/requirements.txt")
    
    # Create a tar.gz file
    print(f"Creating model.tar.gz...")
    with tarfile.open(f"{tmpdirname}/model.tar.gz", "w:gz") as tar:
        tar.add(f"{tmpdirname}/model", arcname=".")
    
    # Upload the tar.gz file to S3
    s3_quantized_model_prefix = f"models/{quantized_model_name}"
    s3_quantized_model_uri = f"s3://{S3_BUCKET}/{s3_quantized_model_prefix}"
    print(f"Uploading model.tar.gz to {s3_quantized_model_uri}...")
    !aws s3 cp {tmpdirname}/model.tar.gz {s3_quantized_model_uri}/model.tar.gz
    
    print(f"Quantized model prepared and uploaded to {s3_quantized_model_uri}/model.tar.gz")

## Deploy Quantized Model to SageMaker Endpoint

Now, let's deploy our quantized model to a SageMaker endpoint for inference.

In [ ]:
# Create HuggingFace model with supported versions and custom inference script
huggingface_model = HuggingFaceModel(
    model_data=f"{s3_quantized_model_uri}/model.tar.gz",
    role=SAGEMAKER_ROLE_ARN,
    transformers_version="4.49.0",
    pytorch_version="2.4.0",
    py_version="py311",
    entry_point="inference.py",  # Use our custom inference script
    env={
        'HF_TASK': model_data.get('task', 'text-classification')
    }
)

In [ ]:
# Create a shorter endpoint name
short_name = "distilbert-base-quantized"  # Much shorter than the full model name
endpoint_name = f"{short_name}-{int(time.time())}"[-63:]  # Ensure it's under 63 chars

# Deploy model to endpoint
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g4dn.xlarge",
    endpoint_name=endpoint_name
)

## Test Inference

Let's test our deployed model with some sample text.

In [ ]:
# Test inference
sample_text = "I really enjoyed this movie. The acting was great and the plot was engaging."

# Measure inference time
start_time = time.time()
response = predictor.predict({
    "inputs": sample_text
})
end_time = time.time()

# Print results
print(f"Inference time: {(end_time - start_time) * 1000:.2f} ms")
print(f"Prediction: {response}")

## Analyze Model Size Reduction

Let's analyze the size reduction achieved through quantization.

In [ ]:
# Get the size of the original model
s3 = boto3.client('s3')
bucket_name = S3_BUCKET
original_model_prefix = model_s3_uri.replace(f"s3://{bucket_name}/", "")
quantized_model_prefix = s3_quantized_model_prefix

# Function to get total size of objects with a prefix
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

# Get sizes
original_size = get_total_size(bucket_name, original_model_prefix)
quantized_size = get_total_size(bucket_name, quantized_model_prefix)

# Convert to MB
original_size_mb = original_size / (1024 * 1024)
quantized_size_mb = quantized_size / (1024 * 1024)

# Calculate reduction
size_reduction = (original_size - quantized_size) / original_size * 100

print(f"Original model size: {original_size_mb:.2f} MB")
print(f"Quantized model size: {quantized_size_mb:.2f} MB")
print(f"Size reduction: {size_reduction:.2f}%")

## Clean Up

When you're done, don't forget to delete the endpoint to avoid incurring charges.

In [ ]:
# Delete endpoint
# Uncomment the line below when you're ready to delete the endpoint
# sagemaker_session.delete_endpoint(endpoint_name)
# print(f"Endpoint {endpoint_name} deleted")